In [3]:
import pandas as pd
import matplotlib.pyplot as plt



In [4]:
# import requests
# import time
# from io import StringIO
# from datetime import datetime

# # --------------------------------------------------
# # SETTINGS
# # --------------------------------------------------

# # Season ending years:
# # 2017 means the 2016–17 NBA season
# START_YEAR = 2017
# END_YEAR = datetime.now().year

# months = [
#     "october",
#     "november",
#     "december",
#     "january",
#     "february",
#     "march",
#     "april",
#     "may",
#     "june"
# ]

# headers = {
#     "User-Agent": (
#         "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
#         "AppleWebKit/537.36 (KHTML, like Gecko) "
#         "Chrome/149.0.0.0 Safari/537.36"
#     )
# }

# all_games = []

# # --------------------------------------------------
# # SCRAPE EVERY SEASON AND MONTH
# # --------------------------------------------------

# for season_end_year in range(START_YEAR, END_YEAR + 1):

#     season_name = f"{season_end_year - 1}-{str(season_end_year)[-2:]}"

#     print(f"\nCollecting {season_name} season...")

#     for month in months:

#         url = (
#             "https://www.basketball-reference.com/"
#             f"leagues/NBA_{season_end_year}_games-{month}.html"
#         )

#         try:
#             response = requests.get(
#                 url,
#                 headers=headers,
#                 timeout=30
#             )

#             # A 404 usually means that month does not exist
#             if response.status_code == 404:
#                 continue

#             response.raise_for_status()

#             tables = pd.read_html(StringIO(response.text))

#             if len(tables) == 0:
#                 continue

#             games = tables[0].copy()

#             # Remove repeated header rows
#             if "Date" in games.columns:
#                 games = games[games["Date"] != "Date"]

#             games["Season"] = season_name
#             games["Season_End_Year"] = season_end_year
#             games["Month"] = month.title()
#             games["Source_URL"] = url

#             all_games.append(games)

#             print(f"  Downloaded {month}: {len(games)} rows")

#             # Pause to avoid sending requests too quickly
#             time.sleep(3)

#         except ValueError:
#             print(f"  No table found for {month}")

#         except requests.RequestException as error:
#             print(f"  Request failed for {month}: {error}")

# # --------------------------------------------------
# # COMBINE ALL MONTHS
# # --------------------------------------------------

# if not all_games:
#     raise ValueError(
#         "No games were collected. Check your internet connection "
#         "or try running the code again later."
#     )

# nba_games = pd.concat(all_games, ignore_index=True)

# # --------------------------------------------------
# # CLEAN COLUMN NAMES
# # --------------------------------------------------

# nba_games.columns = [
#     str(column).strip().replace(" ", "_")
#     for column in nba_games.columns
# ]

# # Basketball Reference sometimes names teams as Visitor/Neutral
# rename_columns = {
#     "Visitor/Neutral": "Away_Team",
#     "Home/Neutral": "Home_Team",
#     "PTS": "Away_PTS",
#     "PTS.1": "Home_PTS",
#     "Attend.": "Attendance",
#     "Arena": "Arena",
#     "Notes": "Notes"
# }

# nba_games = nba_games.rename(columns=rename_columns)

# # --------------------------------------------------
# # CLEAN ATTENDANCE
# # --------------------------------------------------

# if "Attendance" in nba_games.columns:
#     nba_games["Attendance"] = pd.to_numeric(
#         nba_games["Attendance"]
#         .astype(str)
#         .str.replace(",", "", regex=False)
#         .str.strip(),
#         errors="coerce"
#     )

# # Convert scores to numbers
# for column in ["Away_PTS", "Home_PTS"]:

#     if column in nba_games.columns:
#         nba_games[column] = pd.to_numeric(
#             nba_games[column],
#             errors="coerce"
#         )

# # Convert dates
# if "Date" in nba_games.columns:
#     nba_games["Date"] = pd.to_datetime(
#         nba_games["Date"],
#         errors="coerce"
#     )

# # --------------------------------------------------
# # KEEP COMPLETED GAMES WITH ATTENDANCE
# # --------------------------------------------------

# if "Attendance" in nba_games.columns:
#     nba_games_with_attendance = nba_games.dropna(
#         subset=["Attendance"]
#     ).copy()
# else:
#     nba_games_with_attendance = nba_games.copy()
#     print("Warning: An Attendance column was not found.")

# # Sort games chronologically
# if "Date" in nba_games_with_attendance.columns:
#     nba_games_with_attendance = (
#         nba_games_with_attendance
#         .sort_values("Date")
#         .reset_index(drop=True)
#     )

# # --------------------------------------------------
# # SAVE EVERY GAME
# # --------------------------------------------------

# nba_games_with_attendance.to_csv(
#     "nba_every_game_attendance_2017_present.csv",
#     index=False
# )

# print("\nFinished!")
# print(
#     f"Collected {len(nba_games_with_attendance):,} "
#     "games with attendance."
# )

# display(nba_games_with_attendance.head(20))

In [6]:
import pandas as pd
import plotly.graph_objects as go

# --------------------------------------------------
# 1. LOAD BOTH CSV FILES
# --------------------------------------------------

wnba = pd.read_csv("All Game Attendance.csv")
nba = pd.read_csv("nba_every_game_attendance_2017_present.csv")

# --------------------------------------------------
# 2. CLEAN WNBA DATA
# --------------------------------------------------

wnba["Year"] = pd.to_numeric(
    wnba["Year"],
    errors="coerce"
)

wnba["Attendance"] = pd.to_numeric(
    wnba["Attendance"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip(),
    errors="coerce"
)

# Remove 2020 and 2021
wnba = wnba[
    ~wnba["Year"].isin([2020, 2021])
].copy()

# Keep regular-season WNBA games only
wnba_regular = wnba[
    wnba["Game Type"]
        .astype(str)
        .str.strip()
        .str.contains(
            "regular season",
            case=False,
            na=False
        )
].copy()

# Calculate average attendance from individual
# WNBA regular-season games
wnba_yearly = (
    wnba_regular
    .dropna(
        subset=[
            "Year",
            "Attendance"
        ]
    )
    .groupby(
        "Year",
        as_index=False
    )
    .agg(
        WNBA_Average_Attendance=(
            "Attendance",
            "mean"
        ),
        WNBA_Games_Included=(
            "Attendance",
            "count"
        )
    )
)

# --------------------------------------------------
# 3. CLEAN NBA DATA
# --------------------------------------------------

nba["Season_End_Year"] = pd.to_numeric(
    nba["Season_End_Year"],
    errors="coerce"
)

nba["Attendance"] = pd.to_numeric(
    nba["Attendance"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip(),
    errors="coerce"
)

# Remove 2020 and 2021
nba = nba[
    ~nba["Season_End_Year"].isin([2020, 2021])
].copy()

# Make sure the updated scraper created Game_Type
if "Game_Type" not in nba.columns:
    raise ValueError(
        "The NBA CSV does not contain a Game_Type column. "
        "Rerun the updated scraping code before running "
        "this visualization."
    )

# Keep regular-season NBA games only
nba_regular = nba[
    nba["Game_Type"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("regular season")
].copy()

# Calculate average attendance from individual
# NBA regular-season games
nba_yearly = (
    nba_regular
    .dropna(
        subset=[
            "Season_End_Year",
            "Attendance"
        ]
    )
    .groupby(
        "Season_End_Year",
        as_index=False
    )
    .agg(
        NBA_Average_Attendance=(
            "Attendance",
            "mean"
        ),
        NBA_Games_Included=(
            "Attendance",
            "count"
        )
    )
    .rename(
        columns={
            "Season_End_Year": "Year"
        }
    )
)

# --------------------------------------------------
# 4. COMBINE BOTH DATASETS
# --------------------------------------------------

attendance_comparison = pd.merge(
    nba_yearly,
    wnba_yearly,
    on="Year",
    how="inner"
)

attendance_comparison["Year"] = pd.to_numeric(
    attendance_comparison["Year"],
    errors="coerce"
)

attendance_comparison = (
    attendance_comparison
    .dropna(
        subset=[
            "Year",
            "NBA_Average_Attendance",
            "WNBA_Average_Attendance"
        ]
    )
    .sort_values("Year")
    .reset_index(drop=True)
)

attendance_comparison["Year"] = (
    attendance_comparison["Year"].astype(int)
)

# Stop with a clear error if no matching years were found
if attendance_comparison.empty:
    raise ValueError(
        "No matching NBA and WNBA years were found "
        "after cleaning the datasets."
    )

# Round averages for easier reading
attendance_comparison["NBA_Average_Attendance"] = (
    attendance_comparison[
        "NBA_Average_Attendance"
    ].round(0)
)

attendance_comparison["WNBA_Average_Attendance"] = (
    attendance_comparison[
        "WNBA_Average_Attendance"
    ].round(0)
)

# Display the calculated values and game counts
display(
    attendance_comparison[
        [
            "Year",
            "NBA_Average_Attendance",
            "NBA_Games_Included",
            "WNBA_Average_Attendance",
            "WNBA_Games_Included"
        ]
    ]
)

# --------------------------------------------------
# 5. CREATE THE GRAPH
# --------------------------------------------------

fig = go.Figure()

# NBA line
fig.add_trace(
    go.Scatter(
        x=attendance_comparison["Year"],
        y=attendance_comparison[
            "NBA_Average_Attendance"
        ],
        mode="lines+markers",
        name="NBA",
        connectgaps=False,

        line=dict(
            color="#0072B2",
            width=2.5
        ),

        marker=dict(
            color="#0072B2",
            symbol="circle",
            size=6
        ),

        hovertemplate=(
            "<b>NBA</b><br>"
            "Year: %{x}<br>"
            "Average attendance: %{y:,.0f}"
            "<extra></extra>"
        )
    )
)

# WNBA line
fig.add_trace(
    go.Scatter(
        x=attendance_comparison["Year"],
        y=attendance_comparison[
            "WNBA_Average_Attendance"
        ],
        mode="lines+markers",
        name="WNBA",
        connectgaps=False,

        line=dict(
            color="#D55E00",
            width=2.5,
            dash="dash"
        ),

        marker=dict(
            color="#D55E00",
            symbol="square",
            size=6
        ),

        hovertemplate=(
            "<b>WNBA</b><br>"
            "Year: %{x}<br>"
            "Average attendance: %{y:,.0f}"
            "<extra></extra>"
        )
    )
)

# --------------------------------------------------
# 6. LABEL THE FINAL POINTS
# --------------------------------------------------

last_row = attendance_comparison.iloc[-1]

fig.add_annotation(
    x=last_row["Year"],
    y=last_row["NBA_Average_Attendance"],
    text="<b>NBA</b>",
    showarrow=False,
    xanchor="left",
    xshift=8,

    font=dict(
        size=11,
        color="#0072B2"
    )
)

fig.add_annotation(
    x=last_row["Year"],
    y=last_row["WNBA_Average_Attendance"],
    text="<b>WNBA</b>",
    showarrow=False,
    xanchor="left",
    xshift=8,

    font=dict(
        size=11,
        color="#D55E00"
    )
)

# --------------------------------------------------
# 7. FORMAT THE GRAPH
# --------------------------------------------------

minimum_year = attendance_comparison["Year"].min()
maximum_year = attendance_comparison["Year"].max()

fig.update_layout(
    title=dict(
        text=(
            "Average Regular-Season Attendance "
            "per Game: NBA vs. WNBA"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=16)
    ),

    xaxis_title="Year",
    yaxis_title="Average Attendance per Game",

    template="plotly_white",
    hovermode="closest",

    # Smaller graph size
    autosize=False,
    width=700,
    height=420,

    showlegend=False,

    margin=dict(
        l=65,
        r=85,
        t=70,
        b=60
    ),

    font=dict(size=10)
)

fig.update_xaxes(
    range=[
        minimum_year - 0.3,
        maximum_year + 1
    ],

    tickmode="array",

    tickvals=list(
        range(
            minimum_year,
            maximum_year + 1
        )
    ),

    ticktext=[
        str(year)
        for year in range(
            minimum_year,
            maximum_year + 1
        )
    ],

    tickangle=-30,
    tickfont=dict(size=9),
    title_font=dict(size=11),
    gridcolor="#E5ECF6",
    automargin=True
)

fig.update_yaxes(
    tickformat=",",
    rangemode="tozero",
    tickfont=dict(size=9),
    title_font=dict(size=11),
    gridcolor="lightgray",
    automargin=True
)

# --------------------------------------------------
# 8. DISPLAY THE GRAPH
# --------------------------------------------------

fig.show(
    config={
        "responsive": False,
        "displayModeBar": False,
        "displaylogo": False
    }
)

# --------------------------------------------------
# 9. SAVE AS HTML
# --------------------------------------------------

fig.write_html(
    "index.html",
    include_plotlyjs="cdn",
    full_html=True,
    default_width="700px",
    default_height="420px",

    config={
        "responsive": False,
        "displayModeBar": False,
        "displaylogo": False
    }
)



,Year,NBA_Average_Attendance,NBA_Games_Included,WNBA_Average_Attendance,WNBA_Games_Included
0,2017,17884.0,1230,7716.0,204
1,2018,17987.0,1230,6779.0,203
2,2019,17857.0,1230,6528.0,204
3,2022,17170.0,1318,5646.0,216
4,2023,18131.0,1319,6615.0,240
5,2024,18384.0,1319,9807.0,240
6,2025,18185.0,1321,10986.0,286
7,2026,18179.0,1322,11198.0,134
